# gramlab — hybrid tuning (CPU, ~10 min)
Inputs: **gramlab** (Version 3: BM25 index) and **gramlab-map** (Version 2: map + passage matrix). Tunes the BM25 + slot-map rule on half
the scoreboard, reports on the other half, saves the rule next to the map. The output of this notebook carries the map and the rule,
so the chat notebook can attach just this one input (plus gramlab) for the hybrid retriever.

In [ ]:
import os, subprocess, sys, time, glob, shutil, json
T0 = time.time()
def sh(cmd, must=True):
    print('\n$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if must and r.returncode != 0:
        raise SystemExit(f'command failed ({r.returncode}): {cmd}')
os.chdir('/kaggle/working')
if not os.path.exists('gramlab'):
    sh('git clone --depth 1 https://github.com/furevych/gramlab.git')
os.chdir('/kaggle/working/gramlab')
bm = sorted(glob.glob('/kaggle/input/**/work/bm25/weights.npz', recursive=True))
mp = sorted(glob.glob('/kaggle/input/**/results/map2.json', recursive=True))
if not bm or not mp:
    raise SystemExit('Attach both the gramlab (V3) and gramlab-map (V2) outputs as inputs')
os.makedirs('work', exist_ok=True); os.makedirs('results', exist_ok=True)
if not os.path.exists('work/bm25'):
    shutil.copytree(bm[0].split('/weights.npz')[0], 'work/bm25')
mroot = mp[0].split('/results/')[0]
if not os.path.exists('work/map2'):
    shutil.copytree(f'{mroot}/work/map2', 'work/map2')
shutil.copy(mp[0], 'results/')
try:
    import spacy; spacy.load('en_core_web_sm')
except Exception:
    sh('pip install -q spacy && python -m spacy download -q en_core_web_sm', must=False)
best = json.load(open('results/map2.json'))['best_for_hybrid']
sh(f'python scripts/hybrid_tune.py --index work/bm25 --map work/map2/{best} --queries work/bm25/eval_queries.jsonl --results results/hybrid.json')
print(json.dumps(json.load(open('results/hybrid.json'))['chosen'], indent=1))
shutil.rmtree('.git', ignore_errors=True)
print(f'total {(time.time()-T0)/60:.0f} min')